# Neural Network Forecast-Error Correction for ECB SPF Forecasts

This notebook trains the neural networks on the final SPF modelling dataset.

The target is:

```text
forecast_error = actual_value - rolling_1y_forecast
```

The neural network predicts `forecast_error`. The corrected forecast is:

```text
corrected_forecast = rolling_1y_forecast + predicted_error
```

Training setup: feed-forward networks, validation-based hyperparameter
selection, early stopping, and seed ensembles, refit once per test year on an
expanding window.

## Performance notes

The dataset is small by deep-learning standards, so the bottleneck is the number
of refits (architectures x test years x hyperparameters x ensemble seeds), not
data transfer. To keep a full run fast on a Colab GPU:

- no `DataLoader`
- tensors are moved to the selected device before each refit
- mini-batches are created by shuffling tensor indices on-device
- CUDA uses TF32 matmul when available

The SPF dataset has missing values in some predictors, so each
train/validation/test split is first imputed and scaled using the training
sample only; only the clean numeric matrices become tensors.

---
## Step 0: Install Dependencies & Imports

In [1]:
from pathlib import Path
import warnings
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from scipy import stats

warnings.filterwarnings("ignore")

HAS_CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda")

print(f"CUDA: {'Y' if HAS_CUDA else 'N'}")
if not HAS_CUDA:
    raise RuntimeError("CUDA is not available. In Colab, enable a GPU runtime before training.")


CUDA: Y


---
## Step 1: Load the Final SPF Modelling Dataset

This notebook is intended to run in Google Colab. It mounts Google Drive and reads the final modelling dataset directly from:

```text
/content/drive/MyDrive/final_model_dataset2.csv
```


In [2]:
try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError(
        "This notebook is configured for Google Colab. Run it in Colab with Google Drive mounted."
    ) from exc

DRIVE_FILE = Path("/content/drive/MyDrive/final_model_dataset2.csv")

drive.mount("/content/drive")

if not DRIVE_FILE.exists():
    raise FileNotFoundError(
        f"Could not find {DRIVE_FILE}. Check the file name and location in Google Drive."
    )

panel = pd.read_csv(DRIVE_FILE)

panel = panel.sort_values(
    ["survey_year", "survey_quarter", "variable", "fct_source"]
).reset_index(drop=True)

print(f"Panel shape: {panel.shape}")


Panel shape: (9810, 178)


---
## Step 2: Identify Feature Columns and Target

The target is `forecast_error`, the realized value minus the SPF rolling
one-year-ahead forecast.

This cell also chooses the modelling information set. Change `FEATURE_SET` to
run the same NN and linear-baseline pipeline on different inputs:

| Feature set | Purpose |
|---|---|
| `error_history` | Only leak-safe historical forecast-error features. |
| `survey_error` | Past errors plus SPF consensus, disagreement, distance-from-consensus, and horizon-shape features. No macro data and no raw SPF anchor. |
| `survey_error_macro` | Same as `survey_error`, plus EA-MD-QD macro predictors. This is the cleanest test of whether macro information helps predict SPF errors. |
| `survey_error_anchor` | Same as `survey_error`, plus the raw rolling-1y SPF forecast. Tests whether the level of the SPF forecast helps explain errors. |
| `survey_error_macro_anchor` | Survey-error features, macro data, and the raw SPF anchor. This is the richest error-correction setup. |
| `macro_only` | Only macro predictors. Tests whether macro data alone can explain forecast errors. |
| `macro_anchor` | Macro predictors plus the raw SPF forecast. Tests whether macro data improves on the SPF level. |
| `anchor_only` | Only the raw rolling-1y SPF forecast. A minimal benchmark. |
| `all_features` | Previous default: all allowed numeric features. |
| `all_features_no_time` | Same as `all_features`, but removes the time trend. |

The important comparison for the assignment is `survey_error` versus
`survey_error_macro`: same error-history and SPF-disagreement information, with
and without macro data.


In [3]:
# ====================================================
# Separate ID columns, target, and feature-set toggles
# ====================================================

# Train separate models for inflation and real GDP growth.
TARGET_VARIABLES = ["HICP", "RGDP"]

# Main target: the correction the NN should learn.
target_col = "forecast_error"

# Change this one line to rerun every later model on a different information set.
# Most useful first run: "survey_error". Then compare it with "survey_error_macro".
FEATURE_SET = "survey_error"

# Numeric columns that should never be model inputs.
# These are either identifiers, outcomes, target transformations, or the variable
# dummy because HICP and RGDP are modelled separately.
base_excluded_feature_cols = {
    "survey_year",
    "survey_quarter",
    "fct_source",
    "actual_value",
    "forecast_error",
    "abs_forecast_error",
    "squared_forecast_error",
    "rolling_2y_forecast",
    "rolling_longer_forecast",
    "next_year_forecast",
    "current_year_forecast",
    "variable_RGDP",
}

numeric_cols = panel.select_dtypes(include="number").columns.tolist()
candidate_numeric_cols = [
    col for col in numeric_cols
    if col not in base_excluded_feature_cols
]

# Feature groups. The groups are defined from column names so the toggle keeps
# working if the final dataset gains or loses macro predictors later.
spf_anchor_cols = ["rolling_1y_forecast"]

consensus_signal_cols = [
    col for col in candidate_numeric_cols
    if col.startswith((
        "consensus_mean_",
        "consensus_median_",
        "forecast_disagreement_",
        "n_forecasters_",
        "distance_from_consensus_",
        "abs_distance_from_consensus_",
    ))
]

horizon_shape_cols = [
    col for col in ["near_term_tilt", "forward_acceleration", "curvature"]
    if col in candidate_numeric_cols
]

past_error_cols = [
    col for col in candidate_numeric_cols
    if "past_error" in col or "past_abs_error" in col
]

time_feature_cols = [
    col for col in ["survey_round_index"]
    if col in candidate_numeric_cols
]

non_macro_cols = set(
    spf_anchor_cols
    + consensus_signal_cols
    + horizon_shape_cols
    + past_error_cols
    + time_feature_cols
)
macro_feature_cols = [
    col for col in candidate_numeric_cols
    if col not in non_macro_cols
]


def ordered_feature_subset(cols: list[str]) -> list[str]:
    """Return available columns once, preserving final-dataset column order."""
    wanted = set(cols)
    return [col for col in candidate_numeric_cols if col in wanted]


survey_error_cols = ordered_feature_subset(
    past_error_cols + consensus_signal_cols + horizon_shape_cols
)
survey_error_anchor_cols = ordered_feature_subset(
    survey_error_cols + spf_anchor_cols
)
survey_error_macro_cols = ordered_feature_subset(
    survey_error_cols + macro_feature_cols
)
survey_error_macro_anchor_cols = ordered_feature_subset(
    survey_error_cols + macro_feature_cols + spf_anchor_cols
)
macro_anchor_cols = ordered_feature_subset(macro_feature_cols + spf_anchor_cols)
all_features_no_time_cols = [
    col for col in candidate_numeric_cols
    if col not in set(time_feature_cols)
]

feature_set_options = {
    "error_history": ordered_feature_subset(past_error_cols),
    "survey_error": survey_error_cols,
    "survey_error_macro": survey_error_macro_cols,
    "survey_error_anchor": survey_error_anchor_cols,
    "survey_error_macro_anchor": survey_error_macro_anchor_cols,
    "macro_only": ordered_feature_subset(macro_feature_cols),
    "macro_anchor": macro_anchor_cols,
    "anchor_only": ordered_feature_subset(spf_anchor_cols),
    "all_features": candidate_numeric_cols,
    "all_features_no_time": all_features_no_time_cols,
}

if FEATURE_SET not in feature_set_options:
    valid = ", ".join(feature_set_options)
    raise ValueError(f"Unknown FEATURE_SET={FEATURE_SET!r}. Choose one of: {valid}")

feature_cols = feature_set_options[FEATURE_SET]
if not feature_cols:
    raise ValueError(f"FEATURE_SET={FEATURE_SET!r} selected zero columns.")

model_data = panel.copy()

feature_group_summary = pd.DataFrame([
    {"group": "past_error", "n_columns": len(past_error_cols)},
    {"group": "consensus_disagreement", "n_columns": len(consensus_signal_cols)},
    {"group": "horizon_shape", "n_columns": len(horizon_shape_cols)},
    {"group": "raw_spf_anchor", "n_columns": len(spf_anchor_cols)},
    {"group": "macro", "n_columns": len(macro_feature_cols)},
    {"group": "time_trend", "n_columns": len(time_feature_cols)},
])

print(f"Feature set: {FEATURE_SET}")
print(f"Total input features: {len(feature_cols)}")
print("\nAvailable feature groups:")
print(feature_group_summary.to_string(index=False))
print("\nSelected columns:")
print(feature_cols)


Total input features: 156


---
## Step 3: Define Sample Splits

The split must be chronological. A random split would leak information from
future macroeconomic periods into the training sample.

The setup uses an expanding training window, a four-year validation window, and
one test year at a time. Example for test year 2015: train = 2000-2010,
valid = 2011-2014, test = 2015; the next refit shifts everything one year forward.

In [4]:
# ====================================================
# Expanding-window split helpers
# ====================================================

VALID_YEARS = 4
TEST_START_YEAR = 2012


# The panel is sorted by year, so each year is one contiguous block of rows.
# Slices instead of boolean masks keep the repeated annual refits cheap.
def build_year_slices(data: pd.DataFrame) -> dict[int, slice]:
    years = data["survey_year"].to_numpy(copy=True)
    unique_years, first_idx = np.unique(years, return_index=True)

    slices = {}
    for idx, year in enumerate(unique_years):
        start = int(first_idx[idx])
        end = int(first_idx[idx + 1]) if idx + 1 < len(first_idx) else len(years)
        slices[int(year)] = slice(start, end)
    return slices


# One split per test year. For test year 2015: train = start-2010,
# valid = 2011-2014, test = 2015.
def build_splits_for_data(data: pd.DataFrame) -> list[dict]:
    train_start_year = int(data["survey_year"].min())
    test_end_year = int(data["survey_year"].max())

    splits = []
    for test_year in range(TEST_START_YEAR, test_end_year + 1):
        train_end_year = test_year - VALID_YEARS - 1
        valid_start_year = train_end_year + 1
        valid_end_year = test_year - 1

        if train_end_year < train_start_year:
            continue

        splits.append({
            "train": (train_start_year, train_end_year),
            "valid": (valid_start_year, valid_end_year),
            "test":  (test_year, test_year),
        })
    return splits


---
## Step 4: Define the Neural Network Architecture

The model is a feed-forward neural network:

```text
Input -> [Linear -> BatchNorm -> ReLU -> Dropout] x L -> Linear -> predicted_error
```

The output layer has one neuron because the target is one continuous number:
the SPF forecast error. Dropout with `p=0.2` is added after each hidden-layer
ReLU to reduce overfitting in the smaller HICP/RGDP training samples.

In [5]:
# Forecast-error NN

# MLP mapping the feature vector to one number: the predicted correction to the
# rolling 1y SPF forecast.
# Input -> [Linear -> BatchNorm -> ReLU -> Dropout] x L -> Linear(1)
# BatchNorm earns its place here: the macro features have very different scales
# even after standardization. Dropout 0.2 was enough - higher values made the
# small nets underfit.
class SPFNet(nn.Module):

    def __init__(self, input_dim: int, hidden_sizes: list[int]):
        super().__init__()
        layers = []
        prev_dim = input_dim

        for h in hidden_sizes:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(p=0.2))
            prev_dim = h

        layers.append(nn.Linear(prev_dim, 1))  # one output: the error is a single number
        self.network = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x).squeeze(-1)  # (n,1) -> (n,) so it matches y


# Architecture

NN_ARCHITECTURES = {
    "NN1": [32],
    "NN2": [32, 16],
    "NN3": [32, 16, 8],
    "NN4": [32, 16, 8, 4],
    "NN5": [32, 16, 8, 4, 2],
}


---
## Step 5: Training Utilities

Adam optimizer, MSE loss, optional L1 penalty, and early stopping on validation loss.

In [6]:
# Training funktions 

TRAIN_LOSS = nn.MSELoss(reduction="none")


# Mean loss, optionally weighted. The weights exist because one survey round can
# have 50 forecasters and another 25 - without them the busy rounds dominate.
def weighted_loss(
    preds: torch.Tensor,
    target: torch.Tensor,
    sample_weights: torch.Tensor | None = None,
) -> torch.Tensor:
    loss_per_obs = TRAIN_LOSS(preds, target).reshape(-1)
    if sample_weights is None:
        return loss_per_obs.mean()

    weights = sample_weights.reshape(-1).to(loss_per_obs.device)
    denominator = torch.clamp(weights.sum(), min=1e-12)
    return (loss_per_obs * weights).sum() / denominator
# Apply L1 only to Linear layer weights, not BatchNorm.
def select_l1_parameters(model: nn.Module) -> list[torch.Tensor]:
    params = []
    for module in model.modules():
        if isinstance(module, nn.Linear):
            params.append(module.weight)
    return params


# Compute the L1 norm over a precomputed parameter list.
def l1_penalty(parameters: list[torch.Tensor]) -> torch.Tensor:
    l1 = None
    for param in parameters:
        term = param.abs().sum()
        l1 = term if l1 is None else l1 + term
    if l1 is None:
        return torch.tensor(0.0, device=DEVICE)
    return l1


# Clone model weights onto CPU so they can be reused later.
def snapshot_state_dict(model: nn.Module) -> dict[str, torch.Tensor]:
    return {
        name: tensor.detach().cpu().clone()
        for name, tensor in model.state_dict().items()
    }


# Run inference on a tensor that is already on the target device.
#
# Returns a tensor on the SAME device as the input X.
def predict_in_batches(
    model: nn.Module,
    X: torch.Tensor,
    batch_size: int,
    device: torch.device | None = None,
) -> torch.Tensor:
    if X.shape[0] == 0:
        return torch.empty(0, dtype=torch.float32, device=X.device)

    preds = []
    model.eval()
    with torch.inference_mode():
        for start in range(0, X.shape[0], batch_size):
            stop = start + batch_size
            preds.append(model(X[start:stop]))

    return torch.cat(preds, dim=0)


# Evaluate the validation loss used for early stopping and HP tuning.
def evaluate_validation_loss(
    model: nn.Module,
    X: torch.Tensor,
    y: torch.Tensor,
    batch_size: int,
    sample_weights: torch.Tensor | None = None,
) -> float:
    preds = predict_in_batches(model, X, batch_size=batch_size)
    return weighted_loss(preds, y, sample_weights).item()




---
## Step 6: Train Engine

This is the core training routine for one neural network. It takes the prepared train and validation tensors, updates the model weights with Adam, uses validation loss for early stopping, and returns the best version of the model.


In [7]:
# ====================================================
# Train a single neural network with early stopping
# ====================================================

# Train one network with Adam + early stopping on validation loss.
# there is no DataLoader and no per-batch host-to-device copy.
# Returns the restored best model and its validation loss.
def train_single_nn(
    model: nn.Module,
    X_train: torch.Tensor,
    y_train: torch.Tensor,
    X_valid: torch.Tensor,
    y_valid: torch.Tensor,
    train_weights: torch.Tensor | None = None,
    valid_weights: torch.Tensor | None = None,
    lr: float = 0.001,
    l1_lambda: float = 1e-5,
    batch_size: int = 10_000,
    max_epochs: int = 100,
    patience: int = 5,
    inference_batch_size: int | None = None,
) -> tuple[nn.Module, float]:
    if inference_batch_size is None:
        inference_batch_size = INFERENCE_BATCH_SIZE

    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    l1_params = select_l1_parameters(model)

    n_train = X_train.shape[0]
    best_val_loss = float("inf")
    best_state = None
    epochs_no_improve = 0

    for epoch in range(max_epochs):
        model.train()
        # Shuffle indices on-device; no host round-trip.
        perm = torch.randperm(n_train, device=X_train.device)

        # BatchNorm needs at least two observations per training minibatch.
        # If the final tail would have one row, merge it into the previous batch.
        start = 0
        while start < n_train:
            stop = min(start + batch_size, n_train)
            if n_train - stop == 1:
                stop = n_train

            idx = perm[start:stop]
            X_batch = X_train[idx]
            y_batch = y_train[idx]
            w_batch = train_weights[idx] if train_weights is not None else None

            optimizer.zero_grad(set_to_none=True)
            preds = model(X_batch)
            loss = weighted_loss(preds, y_batch, w_batch)
            if l1_lambda:
                loss = loss + l1_lambda * l1_penalty(l1_params)
            loss.backward()
            optimizer.step()

            start = stop

        val_loss = evaluate_validation_loss(
            model,
            X_valid,
            y_valid,
            batch_size=inference_batch_size,
            sample_weights=valid_weights,
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = snapshot_state_dict(model)
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_val_loss


---
## Step 7: Hyperparameter Tuning via Validation

The neural network has several choices that affect how it learns, especially the learning rate and the strength of L1 regularization. Instead of choosing these by looking at the test years, the notebook chooses them inside each refit using only the validation window.

For each architecture and each annual refit, the notebook searches over a small grid of learning rates and L1 penalty strengths.


In [8]:
# ====================================================
# Hyperparameter grid
# ====================================================

FULL_HYPERPARAM_GRID = [
    {"lr": 0.001, "l1_lambda": 1e-5},
    {"lr": 0.01,  "l1_lambda": 1e-5},
    {"lr": 0.001, "l1_lambda": 1e-3},
    {"lr": 0.01,  "l1_lambda": 1e-3},
    {"lr": 0.001, "l1_lambda": 1e-2},
    {"lr": 0.01,  "l1_lambda": 1e-2},
]

HYPERPARAM_GRID = FULL_HYPERPARAM_GRID
MAX_EPOCHS = 100
PATIENCE = 6
N_ENSEMBLE = 10

# Training constants.
BATCH_SIZE = 150
INFERENCE_BATCH_SIZE = 8000

# Numerical-stability guards for train-only preprocessing.
# Some early-window predictors are almost constant after imputation;
# without a std floor, later observations can be divided by ~0.
MIN_FEATURE_STD = 1e-6

# Sample-weight toggle.
# When True, each variable x survey_round x rolling_1y_target block receives
# equal total training/validation weight. This prevents years/periods with
# more forecaster responses from dominating the loss simply because there
# are more rows for the same macro forecast episode.
USE_TARGET_PERIOD_WEIGHTS = True
TARGET_PERIOD_WEIGHT_COLS = ["variable", "survey_round", "rolling_1y_target"]

# Robust target toggle. Bounds are estimated from the train split only,
# then applied to train/validation targets used for learning and tuning.
# Test targets stay unmodified for honest out-of-sample evaluation.
WINSORIZE_TARGET = True
TARGET_WINSOR_LOWER_Q = 0.05
TARGET_WINSOR_UPPER_Q = 0.95

# OLS shrinkage calibrates the correction magnitude on the validation window:
# alpha = sum(pred_valid * error_valid) / sum(pred_valid ** 2), clamped to [0, 1].
USE_OLS_SHRINKAGE = True
SHRINKAGE_ALPHA_MIN = 0.01
SHRINKAGE_ALPHA_MAX = 0.99



---
## Step 8: Data Preparation Helpers

Each annual refit needs the same data-preparation recipe: subset one target variable, build chronological splits, fit imputation and scaling on the train window only, winsorize train/validation targets, and convert arrays to PyTorch tensors.


In [9]:
def to_device_tensor(array: np.ndarray) -> torch.Tensor:
    return torch.from_numpy(array).contiguous().to(
        DEVICE,
        non_blocking=(DEVICE.type == "cuda"),
    )


def build_variable_data(variable: str):
    variable_data = (
        model_data
        .loc[model_data["variable"] == variable]
        .sort_values(["survey_year", "survey_quarter", "fct_source"])
        .copy()
    )
    year_slices = build_year_slices(variable_data)
    splits = build_splits_for_data(variable_data)
    return variable_data, year_slices, splits


def fit_ols_shrinkage_alpha(y_valid_raw: np.ndarray, pred_valid: np.ndarray) -> float:
    if not USE_OLS_SHRINKAGE:
        return 1.0

    y = np.asarray(y_valid_raw, dtype=np.float64)
    p = np.asarray(pred_valid, dtype=np.float64)
    alpha = float(np.dot(p, y) / np.dot(p, p))
    return float(np.clip(alpha, SHRINKAGE_ALPHA_MIN, SHRINKAGE_ALPHA_MAX))


def prepare_split(variable_data: pd.DataFrame, year_slices: dict, split: dict):
    def rows(year_range: tuple[int, int]) -> pd.DataFrame:
        start_year, end_year = year_range
        return variable_data.iloc[
            year_slices[start_year].start:year_slices[end_year].stop
        ]

    train_df = rows(split["train"])
    valid_df = rows(split["valid"])
    test_df = rows(split["test"])

    X_train_df = train_df[feature_cols]
    medians = X_train_df.median(axis=0).fillna(0.0)
    X_train_filled = X_train_df.fillna(medians)
    means = X_train_filled.mean(axis=0).fillna(0.0)
    raw_stds = X_train_filled.std(axis=0, ddof=0).fillna(0.0)
    stds = raw_stds.mask(raw_stds < MIN_FEATURE_STD, 1.0)

    def transform_features(df: pd.DataFrame) -> torch.Tensor:
        scaled = (df[feature_cols].fillna(medians) - means) / stds
        return to_device_tensor(scaled.to_numpy(dtype=np.float32, copy=True))

    def target_weights(df: pd.DataFrame) -> torch.Tensor | None:
        if not USE_TARGET_PERIOD_WEIGHTS:
            return None

        counts = (
            df
            .groupby(TARGET_PERIOD_WEIGHT_COLS, observed=True)[target_col]
            .transform("size")
            .astype(np.float32)
        )
        weights = 1.0 / counts.to_numpy(dtype=np.float32, copy=True)
        weights = weights / weights.mean()
        return to_device_tensor(weights.astype(np.float32, copy=False))

    y_train_raw = train_df[target_col]
    y_valid_raw = valid_df[target_col]
    y_test_raw = test_df[target_col]

    if WINSORIZE_TARGET:
        lower = float(y_train_raw.quantile(TARGET_WINSOR_LOWER_Q))
        upper = float(y_train_raw.quantile(TARGET_WINSOR_UPPER_Q))
        y_train_model = y_train_raw.clip(lower, upper)
        y_valid_model = y_valid_raw.clip(lower, upper)
    else:
        y_train_model = y_train_raw
        y_valid_model = y_valid_raw

    return (
        transform_features(train_df),
        to_device_tensor(y_train_model.to_numpy(dtype=np.float32, copy=True)),
        target_weights(train_df),
        transform_features(valid_df),
        to_device_tensor(y_valid_model.to_numpy(dtype=np.float32, copy=True)),
        target_weights(valid_df),
        transform_features(test_df),
        to_device_tensor(y_test_raw.to_numpy(dtype=np.float32, copy=True)),
        y_valid_raw.to_numpy(dtype=np.float32, copy=True),
        test_df.index.to_numpy(),
    )


---
## Step 9: Full Training Pipeline

For each architecture and each annual refit:

1. select the best hyperparameters using validation MSE
2. train an ensemble with the best hyperparameters
3. average ensemble predictions for the validation and test windows
4. estimate validation-window OLS shrinkage and apply it to test predictions
5. store the calibrated out-of-sample forecast-error predictions


In [10]:
# ====================================================
# Core training pipeline for one architecture
# ====================================================

def train_architecture(
    variable: str,
    arch_name: str,
    hidden_sizes: list[int],
    variable_data: pd.DataFrame,
    year_slices: dict,
    splits: list[dict],
) -> dict:
    all_preds = []
    all_actuals = []
    all_test_years = []
    all_test_indices = []
    input_dim = len(feature_cols)

    print(f"\n{'='*60}")
    print(f"Training {variable} - {arch_name}: hidden layers = {hidden_sizes}")
    print(f"{'='*60}")

    for split in splits:
        test_year = split["test"][0]
        t0 = time.time()

        (
            X_train,
            y_train,
            train_weights,
            X_valid,
            y_valid,
            valid_weights,
            X_test,
            y_test,
            valid_errors_raw,
            test_original_index,
        ) = prepare_split(variable_data, year_slices, split)

        if X_test.shape[0] == 0:
            print(f"  Year {test_year}: no test data, skipping.")
            continue

        best_hp = None
        best_val_loss = float("inf")

        for hp in HYPERPARAM_GRID:
            torch.manual_seed(0)
            np.random.seed(0)
            model = SPFNet(input_dim, hidden_sizes)
            model, val_loss = train_single_nn(
                model,
                X_train,
                y_train,
                X_valid,
                y_valid,
                train_weights=train_weights,
                valid_weights=valid_weights,
                lr=hp["lr"],
                l1_lambda=hp["l1_lambda"],
                batch_size=BATCH_SIZE,
                max_epochs=MAX_EPOCHS,
                patience=PATIENCE,
                inference_batch_size=INFERENCE_BATCH_SIZE,
            )

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_hp = hp

            del model

        ensemble_test_preds = torch.zeros(
            X_test.shape[0], dtype=torch.float32, device=DEVICE
        )
        ensemble_valid_preds = torch.zeros(
            X_valid.shape[0], dtype=torch.float32, device=DEVICE
        )

        for seed in range(N_ENSEMBLE):
            torch.manual_seed(seed)
            np.random.seed(seed)
            model = SPFNet(input_dim, hidden_sizes)
            model, _ = train_single_nn(
                model,
                X_train,
                y_train,
                X_valid,
                y_valid,
                train_weights=train_weights,
                valid_weights=valid_weights,
                lr=best_hp["lr"],
                l1_lambda=best_hp["l1_lambda"],
                batch_size=BATCH_SIZE,
                max_epochs=MAX_EPOCHS,
                patience=PATIENCE,
                inference_batch_size=INFERENCE_BATCH_SIZE,
            )

            ensemble_test_preds += predict_in_batches(
                model,
                X_test,
                batch_size=INFERENCE_BATCH_SIZE,
            )
            ensemble_valid_preds += predict_in_batches(
                model,
                X_valid,
                batch_size=INFERENCE_BATCH_SIZE,
            )
            del model

        ensemble_test_preds /= N_ENSEMBLE
        ensemble_valid_preds /= N_ENSEMBLE

        raw_test_preds = ensemble_test_preds.detach().cpu().numpy()
        raw_valid_preds = ensemble_valid_preds.detach().cpu().numpy()
        alpha = fit_ols_shrinkage_alpha(valid_errors_raw, raw_valid_preds)
        calibrated_test_preds = (alpha * raw_test_preds).astype(np.float32, copy=False)

        all_preds.append(calibrated_test_preds)
        all_actuals.append(y_test.detach().cpu().numpy())
        all_test_years.extend([test_year] * X_test.shape[0])
        all_test_indices.extend(test_original_index.tolist())

        elapsed = time.time() - t0
        print(
            f"  Year {test_year}: "
            f"train={X_train.shape[0]:>6,}, "
            f"valid={X_valid.shape[0]:>6,}, "
            f"test={X_test.shape[0]:>5,} | "
            f"best HP: lr={best_hp['lr']}, lambda1={best_hp['l1_lambda']} | "
            f"alpha={alpha:.3f} | "
            f"{elapsed:.0f}s"
        )

        del X_train, y_train, train_weights, X_valid, y_valid, valid_weights, X_test, y_test
        del ensemble_test_preds, ensemble_valid_preds

    return {
        "variable": variable,
        "preds": np.concatenate(all_preds),
        "actuals": np.concatenate(all_actuals),
        "test_years": np.array(all_test_years),
        "test_indices": np.array(all_test_indices),
    }


---
## Step 10: Run Training

Separate models are trained for HICP inflation and real GDP growth because the two targets have different error distributions.


In [ ]:
# ====================================================
# Select which architectures to train
# ====================================================

archs_to_run = ["NN1", "NN2", "NN3", "NN4", "NN5"]

# Results are nested as results[variable][architecture].
results = {}

total_start = time.time()

for target_variable in TARGET_VARIABLES:
    variable_data, year_slices, splits = build_variable_data(target_variable)
    results[target_variable] = {}

    for arch_name in archs_to_run:
        hidden_sizes = NN_ARCHITECTURES[arch_name]
        results[target_variable][arch_name] = train_architecture(
            target_variable,
            arch_name,
            hidden_sizes,
            variable_data,
            year_slices,
            splits,
        )

total_elapsed = time.time() - total_start
print(f"\nTotal training time: {total_elapsed / 60:.1f} minutes")



Training HICP - NN1: hidden layers = [32]
  Year 2012: train= 1,641, valid=   783, test=  185 | best HP: lr=0.01, lambda1=0.01 | alpha=0.782 | 13s
  Year 2013: train= 1,843, valid=   766, test=  175 | best HP: lr=0.01, lambda1=0.01 | alpha=0.990 | 8s
  Year 2014: train= 2,039, valid=   745, test=  185 | best HP: lr=0.01, lambda1=0.01 | alpha=0.990 | 6s
  Year 2015: train= 2,231, valid=   738, test=  188 | best HP: lr=0.01, lambda1=0.01 | alpha=0.010 | 4s
  Year 2016: train= 2,424, valid=   733, test=  168 | best HP: lr=0.001, lambda1=0.01 | alpha=0.010 | 4s
  Year 2017: train= 2,609, valid=   716, test=  189 | best HP: lr=0.001, lambda1=0.01 | alpha=0.010 | 4s
  Year 2018: train= 2,784, valid=   730, test=  188 | best HP: lr=0.001, lambda1=0.01 | alpha=0.010 | 4s
  Year 2019: train= 2,969, valid=   733, test=  178 | best HP: lr=0.01, lambda1=0.01 | alpha=0.714 | 7s
  Year 2020: train= 3,157, valid=   723, test=  198 | best HP: lr=0.001, lambda1=0.01 | alpha=0.392 | 8s
  Year 2021: tra

---
## Linear baselines — OLS, Ridge, Lasso

Before judging the neural network we fit three linear models on the **same task,
data, splits, preprocessing and calibration** as the NN. Each predicts the SPF
`forecast_error`; the corrected forecast is `SPF forecast + predicted error`.

* **OLS** — ordinary least squares. The "linear, no regularization" reference.
* **Ridge** — L2-penalized; shrinks coefficients smoothly, the standard cure for
  the multicollinearity among the 156 macro features.
* **Lasso** — L1-penalized; drives coefficients to zero, doubling as feature
  selection. Penalty `alpha` is validation-tuned.

> **Heads-up on OLS.** With 156 highly collinear predictors OLS is ill-conditioned
> and extrapolates wildly out of sample, so the validation shrinkage (given a zero
> floor for the linear models) pulls its correction back toward the raw SPF — itself
> a result: unregularized OLS adds little reliable signal.

The two cells below only **train** the linear models into `linear_results`. All
evaluation and graphs live in the **Results** section, where they sit beside the NN.


In [ ]:
# ====================================================
# Linear baselines: shared preprocessing + fit helpers
# ====================================================
# scikit-learn is preinstalled in Colab. The linear models run on CPU, so they
# need numpy arrays rather than the CUDA tensors prepare_split() produces - hence
# the small numpy twin of prepare_split() below. It reuses every global knob the
# NN pipeline uses (feature_cols, MIN_FEATURE_STD, the winsorize/weight toggles)
# so the two model families see the same FEATURE_SET inputs.
from sklearn.linear_model import LinearRegression, Lasso, Ridge


# The NN uses a shrinkage floor of 0.01 (SHRINKAGE_ALPHA_MIN) so its calibrated
# correction never fully vanishes. The linear baselines get their own floor of
# 0.0: ordinary OLS on many correlated predictors can be ill-conditioned
# and extrapolate wildly on test rows, so we let validation shrinkage pull a
# useless correction all the way back to the raw SPF (alpha -> 0) instead of
# leaving a blown-up 0.01 * huge_prediction in place.
LINEAR_SHRINKAGE_ALPHA_MIN = 0.0


def fit_shrinkage_alpha_linear(y_valid_raw: np.ndarray, pred_valid: np.ndarray) -> float:
    if not USE_OLS_SHRINKAGE:
        return 1.0
    y = np.asarray(y_valid_raw, dtype=np.float64)
    p = np.asarray(pred_valid, dtype=np.float64)
    denom = float(np.dot(p, p))
    if denom < 1e-12:                      # degenerate (e.g. Lasso zeroed everything)
        return 0.0
    alpha = float(np.dot(p, y) / denom)
    return float(np.clip(alpha, LINEAR_SHRINKAGE_ALPHA_MIN, SHRINKAGE_ALPHA_MAX))


def prepare_split_numpy(variable_data: pd.DataFrame, year_slices: dict, split: dict) -> dict:
    def rows(year_range: tuple[int, int]) -> pd.DataFrame:
        start_year, end_year = year_range
        return variable_data.iloc[
            year_slices[start_year].start:year_slices[end_year].stop
        ]

    train_df = rows(split["train"])
    valid_df = rows(split["valid"])
    test_df = rows(split["test"])

    # Train-only imputation + standardization (no leakage from valid/test).
    X_train_df = train_df[feature_cols]
    medians = X_train_df.median(axis=0).fillna(0.0)
    X_train_filled = X_train_df.fillna(medians)
    means = X_train_filled.mean(axis=0).fillna(0.0)
    raw_stds = X_train_filled.std(axis=0, ddof=0).fillna(0.0)
    stds = raw_stds.mask(raw_stds < MIN_FEATURE_STD, 1.0)

    def transform_features(df: pd.DataFrame) -> np.ndarray:
        scaled = (df[feature_cols].fillna(medians) - means) / stds
        return scaled.to_numpy(dtype=np.float32, copy=True)

    # Same target-period sample weights as the NN (one survey episode = one unit
    # of weight, regardless of how many forecasters answered).
    def target_weights(df: pd.DataFrame) -> np.ndarray | None:
        if not USE_TARGET_PERIOD_WEIGHTS:
            return None
        counts = (
            df.groupby(TARGET_PERIOD_WEIGHT_COLS, observed=True)[target_col]
            .transform("size")
            .astype(np.float32)
        )
        weights = 1.0 / counts.to_numpy(dtype=np.float32, copy=True)
        return (weights / weights.mean()).astype(np.float32, copy=False)

    y_train_raw = train_df[target_col].to_numpy(dtype=np.float32, copy=True)
    y_valid_raw = valid_df[target_col].to_numpy(dtype=np.float32, copy=True)
    y_test_raw = test_df[target_col].to_numpy(dtype=np.float32, copy=True)

    # Winsorize the *fitting* targets only; test targets stay raw for honest OOS.
    if WINSORIZE_TARGET:
        lower = float(train_df[target_col].quantile(TARGET_WINSOR_LOWER_Q))
        upper = float(train_df[target_col].quantile(TARGET_WINSOR_UPPER_Q))
        y_train_model = np.clip(y_train_raw, lower, upper)
        y_valid_model = np.clip(y_valid_raw, lower, upper)
    else:
        y_train_model = y_train_raw
        y_valid_model = y_valid_raw

    return {
        "X_train": transform_features(train_df),
        "y_train": y_train_model,
        "w_train": target_weights(train_df),
        "X_valid": transform_features(valid_df),
        "y_valid": y_valid_model,
        "w_valid": target_weights(valid_df),
        "X_test": transform_features(test_df),
        "y_test": y_test_raw,
        "y_valid_raw": y_valid_raw,
        "test_index": test_df.index.to_numpy(),
    }


# Some sklearn versions accept sample_weight for Lasso, some do not. Try with,
# fall back to without, so the notebook runs regardless of the installed version.
def _fit_with_optional_weights(estimator, X, y, w):
    if w is None:
        estimator.fit(X, y)
        return estimator
    try:
        estimator.fit(X, y, sample_weight=w)
    except TypeError:
        estimator.fit(X, y)
    return estimator


def _weighted_mse(pred: np.ndarray, y: np.ndarray, w: np.ndarray | None) -> float:
    err2 = (pred - y) ** 2
    if w is None:
        return float(np.mean(err2))
    return float(np.sum(err2 * w) / np.sum(w))


# Penalty grids tried per split; the alpha with the lowest weighted validation
# MSE is kept. This is the linear-model analogue of the NN hyper-parameter grid.
# Lasso (L1) drives coefficients to exactly zero; Ridge (L2) shrinks them
# smoothly and is the natural cure for the collinearity that makes plain OLS
# blow up - hence its much larger penalty range.
LASSO_ALPHA_GRID = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1]
RIDGE_ALPHA_GRID = [1e-1, 1.0, 1e1, 1e2, 1e3, 1e4]


# Validation-tuned fit shared by Lasso and Ridge.
def _tune_penalized(estimator_cls, alpha_grid: list[float], sp: dict, **estimator_kwargs):
    best_model, best_alpha, best_loss = None, None, float("inf")
    for alpha in alpha_grid:
        model = estimator_cls(alpha=alpha, **estimator_kwargs)
        _fit_with_optional_weights(model, sp["X_train"], sp["y_train"], sp["w_train"])
        val_loss = _weighted_mse(model.predict(sp["X_valid"]), sp["y_valid"], sp["w_valid"])
        if val_loss < best_loss:
            best_loss, best_model, best_alpha = val_loss, model, alpha
    return best_model, best_alpha


def fit_linear_estimator(kind: str, sp: dict):
    if kind == "OLS":
        model = LinearRegression()
        _fit_with_optional_weights(model, sp["X_train"], sp["y_train"], sp["w_train"])
        return model, None
    if kind == "Lasso":
        return _tune_penalized(Lasso, LASSO_ALPHA_GRID, sp, max_iter=20_000)
    if kind == "Ridge":
        return _tune_penalized(Ridge, RIDGE_ALPHA_GRID, sp)
    raise ValueError(f"Unknown linear model kind: {kind!r}")


In [ ]:
# ====================================================
# Train OLS and Lasso over the same expanding-window splits
# ====================================================
# Returns the same result dict shape as train_architecture() so the existing
# r2_oos / r2_bias helpers work on it unchanged:
#   {"variable", "preds", "actuals", "test_years", "test_indices"}.
def train_linear_model(
    variable: str,
    kind: str,
    variable_data: pd.DataFrame,
    year_slices: dict,
    splits: list[dict],
) -> dict:
    all_preds, all_actuals, all_years, all_indices = [], [], [], []

    print(f"\n{'='*60}\nFitting {variable} - {kind}\n{'='*60}")

    for split in splits:
        test_year = split["test"][0]
        sp = prepare_split_numpy(variable_data, year_slices, split)

        if sp["X_test"].shape[0] == 0:
            print(f"  Year {test_year}: no test data, skipping.")
            continue

        model, chosen_alpha = fit_linear_estimator(kind, sp)

        raw_valid_preds = model.predict(sp["X_valid"])
        raw_test_preds = model.predict(sp["X_test"])

        # Same validation-based shrinkage calibration the NN uses.
        shrink = fit_shrinkage_alpha_linear(sp["y_valid_raw"], raw_valid_preds)
        calibrated_test = (shrink * raw_test_preds).astype(np.float32, copy=False)

        all_preds.append(calibrated_test)
        all_actuals.append(sp["y_test"])
        all_years.extend([test_year] * sp["X_test"].shape[0])
        all_indices.extend(sp["test_index"].tolist())

        extra = f", {kind.lower()}_alpha={chosen_alpha:g}" if chosen_alpha is not None else ""
        print(
            f"  Year {test_year}: "
            f"train={sp['X_train'].shape[0]:>6,}, "
            f"valid={sp['X_valid'].shape[0]:>6,}, "
            f"test={sp['X_test'].shape[0]:>5,} | "
            f"shrink_alpha={shrink:.3f}{extra}"
        )

    return {
        "variable": variable,
        "preds": np.concatenate(all_preds),
        "actuals": np.concatenate(all_actuals),
        "test_years": np.array(all_years),
        "test_indices": np.array(all_indices),
    }


LINEAR_MODELS = ["OLS", "Ridge", "Lasso"]

# Nested like the NN results: linear_results[variable][model].
linear_results = {}
_lin_start = time.time()

for target_variable in TARGET_VARIABLES:
    variable_data, year_slices, splits = build_variable_data(target_variable)
    linear_results[target_variable] = {}
    for kind in LINEAR_MODELS:
        linear_results[target_variable][kind] = train_linear_model(
            target_variable, kind, variable_data, year_slices, splits
        )

print(f"\nLinear baselines trained in {time.time() - _lin_start:.1f}s")


---
# Results — unified model comparison

Every model produces the **same output**: a predicted correction to the SPF
forecast, stored in the identical schema (`preds`, `actuals`, `test_years`,
`test_indices`). That lets us evaluate OLS, Ridge, Lasso and the neural network
(architectures **NN1, NN3, NN5** and their ensemble average **NN_AVG**) with one
shared set of helpers, metrics and plots.

Two complementary views:

1. **Squared-error view** — R²\_zero (vs raw SPF), R²\_bias (vs a rolling
   mean-error correction) and RMSE.
2. **Absolute-error view — the main hypothesis** —
   `|SPF + model − actual| < |SPF − actual|`: does the correction make the typical
   forecast land closer to the truth? Tested with MAE reduction, win rate and
   one-sided HAC / sign / Wilcoxon tests.

Tables and bar charts show all seven models; the time-series line plots show the
four headline models (OLS, Ridge, Lasso, NN_AVG) to stay readable.


In [ ]:
# ====================================================
# Shared evaluation helpers + unified model registry
# ====================================================
from scipy import stats


# R2 vs leaving the SPF forecast unchanged (squared loss).
def r2_oos(actual_errors, predicted_errors):
    ss_res = np.sum((actual_errors - predicted_errors) ** 2)
    ss_benchmark = np.sum(actual_errors ** 2)
    return 1.0 - ss_res / ss_benchmark


# R2 vs a rolling historical mean-error correction.
def r2_bias(actual_errors, predicted_errors, test_years):
    benchmark = np.zeros_like(actual_errors)
    all_so_far = []
    for yr in np.unique(test_years):
        mask = test_years == yr
        benchmark[mask] = np.mean(all_so_far) if all_so_far else 0.0
        all_so_far.extend(actual_errors[mask].tolist())
    ss_res = np.sum((actual_errors - predicted_errors) ** 2)
    ss_benchmark = np.sum((actual_errors - benchmark) ** 2)
    return 1.0 - ss_res / ss_benchmark


def rmse(values):
    return float(np.sqrt(np.mean(np.asarray(values, dtype=float) ** 2)))


# Newey-West (HAC) mean test on a loss differential; positive stat favors the model.
def newey_west_mean_test(loss_diff, max_lag=4):
    d = np.asarray(loss_diff, dtype=float)
    n = len(d)
    mean_diff = float(np.mean(d))
    demeaned = d - mean_diff
    lag = min(max_lag, n - 1)
    long_run_var = float(np.dot(demeaned, demeaned) / n)
    for k in range(1, lag + 1):
        gamma_k = float(np.dot(demeaned[k:], demeaned[:-k]) / n)
        long_run_var += 2.0 * (1.0 - k / (lag + 1)) * gamma_k
    se = float(np.sqrt(long_run_var / n)) if long_run_var > 0 else 0.0
    stat = mean_diff / se if se > 0 else 0.0
    return {"n": n, "mean_diff": mean_diff, "stat": float(stat),
            "p_two_sided": float(2.0 * stats.norm.sf(abs(stat)))}


# Ensemble-mean NN correction across architectures (aligned by test row).
def average_architecture_result(variable_results, arch_names):
    available = [a for a in arch_names if a in variable_results]
    if len(available) < 2:
        return None
    reference = variable_results[available[0]]
    stacked = [reference["preds"]]
    for a in available[1:]:
        res = variable_results[a]
        if not np.array_equal(res["test_indices"], reference["test_indices"]):
            raise ValueError(f"Test rows not aligned for {a}; cannot build NN_AVG.")
        stacked.append(res["preds"])
    return {"actuals": reference["actuals"],
            "preds": np.mean(np.vstack(stacked), axis=0),
            "test_years": reference["test_years"],
            "test_indices": reference["test_indices"]}


# --- model registry: display order, colors, and which models appear in line plots ---
NN_ARCHES_SHOWN = ["NN1", "NN3", "NN5"]
NN_AVG_NAME = "NN_AVG"
COMPARISON_MODELS = ["OLS", "Ridge", "Lasso", *NN_ARCHES_SHOWN, NN_AVG_NAME]
LINE_PLOT_MODELS = ["OLS", "Ridge", "Lasso", NN_AVG_NAME]

MODEL_COLORS = {
    "OLS": "tab:blue", "Ridge": "tab:purple", "Lasso": "tab:orange",
    "NN1": "tab:green", "NN3": "tab:olive", "NN5": "tab:cyan", "NN_AVG": "tab:red",
}
VARIABLE_TITLES = {"HICP": "HICP Inflation", "RGDP": "Real GDP Growth"}


# Gather every model's result dict (identical schema) for one variable.
# NN_AVG averages all architectures actually trained (archs_to_run).
def collect_model_results(target_variable):
    out = {}
    for name in ["OLS", "Ridge", "Lasso"]:
        out[name] = linear_results[target_variable][name]
    for arch in NN_ARCHES_SHOWN:
        if arch in results.get(target_variable, {}):
            out[arch] = results[target_variable][arch]
    nn_avg = average_architecture_result(results.get(target_variable, {}), archs_to_run)
    if nn_avg is not None:
        out[NN_AVG_NAME] = nn_avg
    return out


In [ ]:
# ====================================================
# Target-period forecast frames (shared by tables and plots)
# ====================================================
TS_META_COLS = ["variable", "rolling_1y_target", "target_date",
                "rolling_1y_forecast", "actual_value"]


def model_pred_series(target_variable, model_name):
    res = collect_model_results(target_variable)[model_name]
    return pd.Series(res["preds"], index=res["test_indices"], name=model_name)


# Average forecaster-level predictions to one point per target period.
def forecast_frame_from_series(pred_series):
    meta = model_data.loc[pred_series.index, TS_META_COLS].reset_index(drop=True)
    meta["predicted_error"] = pred_series.to_numpy(dtype=float)
    meta["corrected_forecast"] = meta["rolling_1y_forecast"] + meta["predicted_error"]
    meta["plot_date"] = pd.to_datetime(meta["target_date"])
    grouped = (meta.groupby(["variable", "rolling_1y_target", "plot_date"], as_index=False)
               .agg(actual_value=("actual_value", "mean"),
                    average_spf_forecast=("rolling_1y_forecast", "mean"),
                    average_corrected_forecast=("corrected_forecast", "mean"))
               .sort_values("plot_date"))
    grouped["raw_error"] = grouped["actual_value"] - grouped["average_spf_forecast"]
    grouped["corrected_error"] = grouped["actual_value"] - grouped["average_corrected_forecast"]
    return grouped


In [ ]:
# ====================================================
# Master metrics table (per forecaster row)
# ====================================================
master_rows = []
for target_variable in TARGET_VARIABLES:
    model_results = collect_model_results(target_variable)
    reference = next(iter(model_results.values()))
    actual_errors = reference["actuals"]
    test_years = reference["test_years"]

    master_rows.append({
        "Variable": target_variable, "Model": "Raw SPF",
        "R2_zero (%)": 0.0,
        "R2_bias (%)": r2_bias(actual_errors, np.zeros_like(actual_errors), test_years) * 100,
        "RMSE": rmse(actual_errors),
        "MAE": float(np.mean(np.abs(actual_errors))),
        "MAE reduction (%)": 0.0,
        "Win rate (%)": np.nan,
        "N": len(actual_errors),
    })
    for name in [m for m in COMPARISON_MODELS if m in model_results]:
        res = model_results[name]
        resid = res["actuals"] - res["preds"]
        raw_abs, corr_abs = np.abs(res["actuals"]), np.abs(resid)
        master_rows.append({
            "Variable": target_variable, "Model": name,
            "R2_zero (%)": r2_oos(res["actuals"], res["preds"]) * 100,
            "R2_bias (%)": r2_bias(res["actuals"], res["preds"], res["test_years"]) * 100,
            "RMSE": rmse(resid),
            "MAE": float(np.mean(corr_abs)),
            "MAE reduction (%)": 100.0 * (np.mean(raw_abs) - np.mean(corr_abs)) / np.mean(raw_abs),
            "Win rate (%)": 100.0 * np.mean(corr_abs < raw_abs),
            "N": len(res["actuals"]),
        })

master_metrics_df = pd.DataFrame(master_rows)
print("=" * 96)
print("MASTER METRICS (per forecaster row). R2/RMSE = squared-error view; MAE/Win = absolute-error view.")
print("=" * 96)
_show = master_metrics_df.copy()
for col in ["R2_zero (%)", "R2_bias (%)", "RMSE", "MAE", "MAE reduction (%)", "Win rate (%)"]:
    _show[col] = _show[col].map(lambda x: "" if pd.isna(x) else f"{x:.3f}")
_show["N"] = _show["N"].map(lambda x: f"{x:,}")
try:
    display(_show)
except NameError:
    print(_show.to_string(index=False))


In [ ]:
# ====================================================
# Hypothesis test:  | SPF + model - actual |  <  | SPF - actual |
# ====================================================
def _sign_test_greater(wins, total):
    if total == 0:
        return float("nan")
    try:
        return float(stats.binomtest(wins, total, 0.5, alternative="greater").pvalue)
    except AttributeError:                       # very old scipy
        mu, sd = total / 2.0, np.sqrt(total) / 2.0
        return float(stats.norm.sf((wins - mu - 0.5) / sd))


def absolute_error_hypothesis(raw_error, corrected_error):
    raw_abs = np.abs(np.asarray(raw_error, dtype=float))
    corr_abs = np.abs(np.asarray(corrected_error, dtype=float))
    gain = raw_abs - corr_abs                    # > 0  <=>  correction made |error| smaller
    n = len(gain)
    nw = newey_west_mean_test(gain)
    wins, losses = int(np.sum(gain > 0)), int(np.sum(gain < 0))
    try:
        wilcoxon_p = float(stats.wilcoxon(raw_abs, corr_abs, alternative="greater").pvalue)
    except ValueError:
        wilcoxon_p = float("nan")
    return {"N": n,
            "MAE raw": float(np.mean(raw_abs)),
            "MAE corrected": float(np.mean(corr_abs)),
            "MAE reduction (%)": 100.0 * (np.mean(raw_abs) - np.mean(corr_abs)) / np.mean(raw_abs),
            "Win rate (%)": 100.0 * wins / n,
            "DM p (1-sided)": float(stats.norm.sf(nw["stat"])),
            "Sign p (1-sided)": _sign_test_greater(wins, wins + losses),
            "Wilcoxon p (1-sided)": wilcoxon_p}


def hypothesis_verdict(mae_reduction_pct, p_one_sided):
    if mae_reduction_pct <= 0:
        return "No (enlarges |error|)"
    return "Yes, significant" if p_one_sided < 0.05 else "Yes, not significant"


hyp_rows = []
for target_variable in TARGET_VARIABLES:
    model_results = collect_model_results(target_variable)
    for name in [m for m in COMPARISON_MODELS if m in model_results]:
        res = model_results[name]
        row = absolute_error_hypothesis(res["actuals"], res["actuals"] - res["preds"])
        frame = forecast_frame_from_series(model_pred_series(target_variable, name))
        period = absolute_error_hypothesis(frame["raw_error"].to_numpy(),
                                           frame["corrected_error"].to_numpy())
        for level, st in [("Per forecaster row", row), ("Per target period", period)]:
            hyp_rows.append({"Variable": target_variable, "Model": name, "Level": level, **st,
                             "H1 supported?": hypothesis_verdict(st["MAE reduction (%)"],
                                                                 st["DM p (1-sided)"])})

hypothesis_df = pd.DataFrame(hyp_rows)
print("=" * 96)
print("HYPOTHESIS:  | (SPF + model) - actual |  <  | SPF - actual |   (one-sided; p < 0.05 = significant)")
print("=" * 96)
_h = hypothesis_df.copy()
for col in ["MAE raw", "MAE corrected", "MAE reduction (%)", "Win rate (%)",
            "DM p (1-sided)", "Sign p (1-sided)", "Wilcoxon p (1-sided)"]:
    _h[col] = _h[col].map(lambda x: f"{x:.4f}")
try:
    display(_h)
except NameError:
    print(_h.to_string(index=False))


In [ ]:
# ====================================================
# Graph 1: MAE and win-rate by model (per forecaster row, all 7 models)
# ====================================================
bar_models = [m for m in COMPARISON_MODELS if m in set(master_metrics_df["Model"])]
x = np.arange(len(TARGET_VARIABLES))
width = 0.8 / max(len(bar_models), 1)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
for j, name in enumerate(bar_models):
    vals = [master_metrics_df.query("Variable == @v and Model == @name")["MAE"].iloc[0]
            for v in TARGET_VARIABLES]
    ax.bar(x + j * width, vals, width, label=name, color=MODEL_COLORS.get(name))
for i, v in enumerate(TARGET_VARIABLES):
    raw_mae = master_metrics_df.query("Variable == @v and Model == 'Raw SPF'")["MAE"].iloc[0]
    ax.hlines(raw_mae, x[i] - 0.1, x[i] - 0.1 + len(bar_models) * width,
              color="black", linestyle="--", linewidth=1.3, label="Raw SPF" if i == 0 else None)
ax.set_xticks(x + width * (len(bar_models) - 1) / 2)
ax.set_xticklabels(TARGET_VARIABLES)
ax.set_ylabel("Mean absolute error")
ax.set_title("MAE by model (below dashed = beats SPF)")
ax.legend(fontsize=8); ax.grid(alpha=0.3, axis="y")

ax = axes[1]
for j, name in enumerate(bar_models):
    vals = [master_metrics_df.query("Variable == @v and Model == @name")["Win rate (%)"].iloc[0]
            for v in TARGET_VARIABLES]
    ax.bar(x + j * width, vals, width, label=name, color=MODEL_COLORS.get(name))
ax.axhline(50, color="black", linestyle="--", linewidth=1.3, label="50% (coin flip)")
ax.set_xticks(x + width * (len(bar_models) - 1) / 2)
ax.set_xticklabels(TARGET_VARIABLES)
ax.set_ylabel("Win rate (% of rows |corrected| < |raw|)")
ax.set_title("How often the correction beats SPF")
ax.legend(fontsize=8); ax.grid(alpha=0.3, axis="y")

fig.suptitle("Absolute-error comparison across all models", y=1.02)
plt.tight_layout(); plt.show()


In [ ]:
# ====================================================
# Graph 2: actual vs raw SPF vs corrected forecasts over time (headline models)
# ====================================================
fig, axes = plt.subplots(len(TARGET_VARIABLES), 1, figsize=(14, 9))
if len(TARGET_VARIABLES) == 1:
    axes = [axes]

for ax, target_variable in zip(axes, TARGET_VARIABLES):
    available = [m for m in LINE_PLOT_MODELS if m in collect_model_results(target_variable)]
    base = forecast_frame_from_series(model_pred_series(target_variable, available[0]))

    ax.plot(base["plot_date"], base["actual_value"], color="black", linewidth=2.4, label="Realized actual")
    ax.plot(base["plot_date"], base["average_spf_forecast"], color="tab:gray", linewidth=1.8,
            linestyle="--", label="Raw SPF forecast")
    for name in available:
        frame = forecast_frame_from_series(model_pred_series(target_variable, name))
        ax.plot(frame["plot_date"], frame["average_corrected_forecast"],
                color=MODEL_COLORS.get(name), linewidth=1.7, label=f"SPF + {name}")

    ax.set_title(f"{VARIABLE_TITLES.get(target_variable, target_variable)} — actual vs forecasts")
    ax.set_ylabel("Percent / annual growth")
    ax.grid(alpha=0.3); ax.legend(fontsize=8)

axes[-1].set_xlabel("Target period")
fig.suptitle("Out-of-sample: actual vs SPF vs corrected forecasts", y=1.01)
plt.tight_layout(); plt.show()


In [ ]:
# ====================================================
# Graph 3: absolute error over time (colored below black = correction helped)
# ====================================================
fig, axes = plt.subplots(len(TARGET_VARIABLES), 1, figsize=(14, 9))
if len(TARGET_VARIABLES) == 1:
    axes = [axes]

for ax, target_variable in zip(axes, TARGET_VARIABLES):
    available = [m for m in LINE_PLOT_MODELS if m in collect_model_results(target_variable)]
    base = forecast_frame_from_series(model_pred_series(target_variable, available[0]))

    ax.plot(base["plot_date"], base["raw_error"].abs(), color="black", linewidth=2.4, label="|raw SPF error|")
    for name in available:
        frame = forecast_frame_from_series(model_pred_series(target_variable, name))
        ax.plot(frame["plot_date"], frame["corrected_error"].abs(),
                color=MODEL_COLORS.get(name), linewidth=1.7, label=f"|{name} corrected|")

    ax.set_title(f"{VARIABLE_TITLES.get(target_variable, target_variable)} — absolute error over time")
    ax.set_ylabel("|forecast error|")
    ax.grid(alpha=0.3); ax.legend(fontsize=8)

axes[-1].set_xlabel("Target period")
fig.suptitle("Absolute error over time", y=1.01)
plt.tight_layout(); plt.show()


In [ ]:
# ====================================================
# Graph 4: year-by-year out-of-sample R2 vs raw SPF (headline models)
# ====================================================
fig, axes = plt.subplots(1, len(TARGET_VARIABLES), figsize=(16, 5), sharey=True)
if len(TARGET_VARIABLES) == 1:
    axes = [axes]

for ax, target_variable in zip(axes, TARGET_VARIABLES):
    model_results = collect_model_results(target_variable)
    for name in [m for m in LINE_PLOT_MODELS if m in model_results]:
        res = model_results[name]
        years = np.unique(res["test_years"])
        yearly = [r2_oos(res["actuals"][res["test_years"] == yr],
                         res["preds"][res["test_years"] == yr]) * 100 for yr in years]
        ax.plot(years, yearly, marker="o", markersize=4, color=MODEL_COLORS.get(name), label=name)

    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_title(f"{VARIABLE_TITLES.get(target_variable, target_variable)} — annual R2 vs SPF")
    ax.set_xlabel("Test year")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

axes[0].set_ylabel("R2_OOS (%)")
fig.suptitle("Year-by-year improvement over the raw SPF", y=1.02)
plt.tight_layout(); plt.show()
